# Advection et conduction dans une colonne nappe-riviere (M2)

Objectifs : comprendre comment un gradient de charge (infiltration ou exfiltration) transporte la chaleur, et comment la porosite, la conductivite thermique et la permeabilite controlent le signal de temperature mesure a quelques decimetres sous le lit.

## 1. Equations

Transport d'energie dans le milieu poreux sature (une dimension, axe $z$ oriente vers le haut) :

$$
\underbrace{\left[\phi C_w + (1-\phi) C_s\right]}_{C_{v}} \frac{\partial T}{\partial t}
= \underbrace{\lambda \frac{\partial^2 T}{\partial z^2}}_{\text{conduction}}
- \underbrace{\phi C_w\, q \frac{\partial T}{\partial z}}_{\text{advection}}
$$

- $\phi$ : porosite, $C_w$ : capacite volumique de l'eau, $C_s$ : capacite specifique du solide.
- $\lambda$ : conductivite thermique du milieu [W m$^{-1}$ K$^{-1}$].
- $q$ : vitesse de Darcy [m/s], positive vers le haut (exfiltration) dans la convention Ginette.

Loi de Darcy, avec la charge $h$ :

$$
q = -\frac{k \rho g}{\mu} \frac{\partial h}{\partial z}
\quad\Rightarrow\quad
q \approx -K \frac{h_{\text{top}} - h_{\text{bottom}}}{L}
$$

**Attention au signe.** Si $h_{\text{top}} > h_{\text{bottom}}$, l'eau descend : c'est une **infiltration** (riviere vers nappe). Si $h_{\text{top}} < h_{\text{bottom}}$, l'eau monte : c'est une **exfiltration** (nappe vers riviere).

## 2. Nombre de Peclet

Le rapport entre advection et conduction sur la longueur $L$ de la colonne :

$$
\mathrm{Pe} = \frac{q\, C_w\, L}{\lambda}
$$

- $|\mathrm{Pe}| \ll 1$ : la conduction domine, le profil de temperature est celui d'un milieu immobile.
- $|\mathrm{Pe}| \gg 1$ : l'advection domine, la chaleur est transportee par l'eau.

Avant de simuler, calculez le nombre de Peclet pour les trois cas du TD.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
sys.path.append(str(Path.cwd().resolve().parents[1] / 'src' / 'src_python'))
from Init_folders import setup_lab_environment
from Direct_model import (setup_ginette, initial_conditions, boundary_conditions,
                          run_direct_model, write_coordonnee_file)
td_dir, synth_dir = setup_lab_environment()
print('Dossier de travail :', synth_dir)


## 3. Geometrie et constantes

Colonne de 5 m, maille de 2 cm (modifiable : `dz` doit diviser 5 m). Les capteurs sont a 10, 20, 30 et 40 cm sous la surface : ils sont places automatiquement en fonction de `dz`.

In [ ]:
z_top, z_bottom = 0.0, -5.0
az = z_top - z_bottom
dz = 0.02
n_depths = round(az / dz)
assert abs(n_depths * dz - az) < 1e-9, 'dz doit diviser la hauteur de colonne'
dz_obs = 0.1
dt = 600
nb_day = 30
state = 1
date_simul_bg = pd.to_datetime('2022/04/21 14:00:00')
mu, rho, g = 1e-3, 1000.0, 9.81
Cw_vol = 4.185e6   # J/m3/K
print(f'{n_depths} mailles de {dz} m, capteurs a {dz_obs} m d\'intervalle')


## 4. Un cas = un jeu de parametres

La fonction ci-dessous construit les conditions aux limites pour un regime donne et lance Ginette. Le regime est fixe par la difference de charge moyenne `dh` (m) entre le haut et le bas de la colonne :
- `dh = 0` : conduction pure (controle)
- `dh > 0` : infiltration
- `dh < 0` : exfiltration

La temperature de surface a un cycle journalier et une tendance, la base est a 10 °C.

In [ ]:
def simuler(dh=0.0, n=0.4, log_k=-10.8, lam=1.0, c_solid=800.0, nb_jours=nb_day):
    t = np.arange(0, nb_jours * 86400, dt)
    df = pd.DataFrame({'times': t, 'days': t / 86400})
    df['T_top'] = 5 * np.sin(2 * np.pi * df['days']) + 0.3 * df['days'] + 17
    df['T_bottom'] = 10.0
    df['h_bottom'] = 5.0
    df['h_top'] = 5.0 + dh
    df['head_gradient'] = dh / az
    k = 10 ** log_k
    K = k * rho * g / mu
    z_obs = setup_ginette(dt, state, nb_jours, z_top, z_bottom, az, dz, date_simul_bg, dz_obs)
    write_coordonnee_file(z_bottom, dz)
    initial_conditions(df, z_top, z_bottom, dz, z_obs)
    boundary_conditions(df, dt)
    sim = run_direct_model(date_simul_bg, z_bottom, dz, 1, 0.0, log_k, n, lam, c_solid)
    flux_eau = pd.read_csv('Sim_velocity_profil_t.dat', sep=r'\s+', header=None, names=['Time', 'z', 'q'])
    flux_chal = pd.read_csv('Sim_heat_flux_profil_t.dat', sep=r'\s+', header=None,
                            names=['Time', 'z', 'Conductive', 'Advective', 'Total'])
    sim.attrs['flux_eau'] = flux_eau
    sim.attrs['flux_chal'] = flux_chal
    q = -K * dh / az            # vitesse de Darcy [m/s], positive vers le haut
    Pe = q * Cw_vol * az / lam
    return sim, q, Pe
print('fonction simuler prete')


### Calcul du nombre de Peclet (a faire avant de simuler)

Etapes : $K = k\rho g/\mu$, puis $q = -K\,\Delta h / L$, puis $\mathrm{Pe} = q\,C_w L/\lambda$. Completez le tableau pour $\Delta h = 0.02$ m et $\Delta h = 0.2$ m, avec les parametres par defaut, puis comparez a la simulation.

In [ ]:
# Etape 1 : conductivite hydraulique
log_k_def, lam_def = -10.8, 1.0
K = 10 ** log_k_def * rho * g / mu
print(f'K = {K:.3e} m/s')

# Etape 2 et 3 : vitesse de Darcy puis Peclet, pour plusieurs differences de charge
for dh_test in [0.005, 0.02, 0.2]:
    q_test = -K * dh_test / az
    Pe_test = q_test * Cw_vol * az / lam_def
    print(f'dh = {dh_test:+.3f} m  ->  q = {q_test:+.2e} m/s ({q_test*86400:+.3f} m/j)   Pe = {Pe_test:+.2f}')


## 5. Trois niveaux de transport pour chaque sens d'ecoulement

Pour chaque sens (infiltrant, exfiltrant), trois cas de nombre de Peclet :
- **conduction** : $|\mathrm{Pe}| = 0.01$, le sens d'ecoulement existe mais la conduction domine ;
- **transition** : $|\mathrm{Pe}| \approx 1$ ;
- **advection** : $|\mathrm{Pe}| \approx 100$.

Avant de regarder les courbes, prevoyez : a 20 cm, quel cas se rapproche le plus de la surface, et lequel est le plus amorti ?

In [ ]:
# Differences de charge calibrees sur le nombre de Peclet (|Pe| = |K*dh*Cw/lam| avec lam = 1)
K_cas = 10 ** -10.8 * rho * g / mu
dh_cond = 0.01 / (K_cas * Cw_vol)         # Pe = 0.01 : conduction dominante, sens fixe
dh_trans = 1.0 / (K_cas * Cw_vol)         # Pe = 1
dh_adv = 100.0 / (K_cas * Cw_vol)         # Pe = 100
scenarios = {
    'Conduction, infiltrant': dh_cond, 'Conduction, exfiltrant': -dh_cond,
    'Transition, infiltrant': dh_trans, 'Transition, exfiltrant': -dh_trans,
    'Advection, infiltrant': dh_adv, 'Advection, exfiltrant': -dh_adv,
}
resultats = {}
for nom, dh_s in scenarios.items():
    sim, q_s, Pe_s = simuler(dh=dh_s)
    resultats[nom] = sim
    print(f'{nom:28s} dh = {dh_s:+.6f} m   q = {q_s:+.2e} m/s   Pe = {Pe_s:+.3f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, sens in zip(axes, ['infiltrant', 'exfiltrant']):
    for regime in ['Conduction', 'Transition', 'Advection']:
        ax.plot(resultats[f'{regime}, {sens}']['Time'] / 86400,
                resultats[f'{regime}, {sens}']['Temp2'], label=regime)
    ax.set_title(f'Capteur a 20 cm - {sens}')
    ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('Temperature [°C]')
plt.tight_layout(); plt.show()


### Sorties Ginette : temperature, flux d'eau et flux de chaleur

- **Temperature** : evolution a 20 cm.
- **Flux d'eau** : vitesse de Darcy $q$ a 20 cm [m/s], positive vers le haut. C'est le debit specifique qui traverse l'interface.
- **Flux de chaleur** : flux conductif $-\lambda\,\partial T/\partial z$ et flux advectif $\phi C_w q T$, en W/m². Leur somme est le flux total. Observez quelle part domine dans chaque regime.

In [ ]:
def serie_profondeur(sim, df, colonne, z_cible=-0.2):
    z_u = df['z'].unique()
    z_proche = z_u[np.argmin(np.abs(z_u - z_cible))]
    d = df[np.isclose(df['z'], z_proche)]
    return d['Time'].values / 86400, d[colonne].values

fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)
for ligne, sens in enumerate(['infiltrant', 'exfiltrant']):
    for regime in ['Conduction', 'Transition', 'Advection']:
        sim = resultats[f'{regime}, {sens}']
        t, T = serie_profondeur(sim, sim.rename(columns={'Time':'Time'}).assign(z=0.0) if False else None, None) if False else (sim['Time'].values/86400, sim['Temp2'].values)
        axes[ligne, 0].plot(t, T, label=regime)
        t, q = serie_profondeur(sim, sim.attrs['flux_eau'], 'q')
        axes[ligne, 1].plot(t, q, label=regime)
        t, adv = serie_profondeur(sim, sim.attrs['flux_chal'], 'Advective')
        t, cond = serie_profondeur(sim, sim.attrs['flux_chal'], 'Conductive')
        axes[ligne, 2].plot(t, cond + adv, label=f'{regime} (total)')
    axes[ligne, 0].set_ylabel(f'{sens}\nTemperature [°C]')
    axes[ligne, 1].set_ylabel('Flux d\'eau q [m/s]')
    axes[ligne, 2].set_ylabel('Flux de chaleur [W/m²]')
for ax, titre in zip(axes[0], ['Temperature a 20 cm', 'Vitesse de Darcy a 20 cm', 'Flux de chaleur total a 20 cm']):
    ax.set_title(titre)
for ax in axes[-1]:
    ax.set_xlabel('Temps [jours]')
for ax in axes.ravel():
    ax.grid(alpha=0.3); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()


### Lire les figures

- **Conduction** : la courbe a 20 cm est presque identique au signal de surface, mais retardee et attenuee par diffusion. Le sens du flux ne change rien de visible : c'est le transport par diffusion, qui est symetrique.
- **Transition** : le sens du flux commence a se voir. Infiltration et exfiltration ne donnent pas la meme courbe, car l'eau transporte la chaleur dans un sens donne.
- **Advection** : le signal de surface arrive presque sans attenuation en infiltration (l'eau froide ou chaude descend vite), alors qu'en exfiltration la nappe, plus stable a 10 °C, impose son niveau et attenue la surface.

Question : pour l'infiltration, a quelle vitesse le signal de surface voyage-t-il ? Estimez-la sur la figure, puis comparez a $|q|/n$ (vitesse reelle de l'eau).

In [ ]:
# Profils verticaux de temperature a un instant donne (apres 20 jours)
i_t = int(20 * 86400 / dt)
fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
for ax, sens in zip(axes, ['infiltrant', 'exfiltrant']):
    for regime in ['Conduction', 'Transition', 'Advection']:
        sim = resultats[f'{regime}, {sens}']
        capteurs = [c for c in sim.columns if c.startswith('Temp')]
        ax.plot([float(sim[c].iloc[i_t]) for c in capteurs],
                [-0.1 * (k + 1) for k in range(len(capteurs))], 'o-', label=regime)
    ax.set_title(f'Profil a 20 jours - {sens}')
    ax.set_xlabel('Temperature [°C]'); ax.grid(alpha=0.3); ax.legend(fontsize=8)
axes[0].set_ylabel('Profondeur [m]')
plt.tight_layout(); plt.show()

Sur le profil, observez : la temperature de la surface penetre-t-elle plus loin en infiltration ? Pourquoi l'exfiltration rapproche-t-elle les profils de la temperature de la base ?

### Synthese

| Regime | |Pe| | Ce qu'il faut retenir |
|---|---|---|
| Conduction | 0.01 | Diffusion seule : retard et attenuation, sens du flux invisible |
| Transition | 1 | Le sens du flux commence a modifier le signal |
| Advection | 100 | L'eau transporte la chaleur : asymetrie infiltration/exfiltration |

Pour aller plus loin : refaites l'exercice avec une porosite $n = 0.2$ puis $n = 0.4$ dans le regime de transition, et expliquez le resultat a l'aide de $C_v$ et de $\lambda$.

## 6. Effet des parametres (regime infiltrant)

Faites varier un parametre a la fois. Le signal a 40 cm s'amortit-il et se decale-t-il plus ou moins ? Pourquoi ?

Pistes :
- porosite $\phi$ : agit sur la capacite $C_v$ et sur la conductivite equivalente ;
- conductivite $\lambda$ : diffusion thermique pure ;
- permeabilite $k$ : vitesse de Darcy, donc nombre de Peclet.


In [ ]:
variations = {'n = 0.2': dict(dh=0.02, n=0.2), 'n = 0.4': dict(dh=0.02, n=0.4),
              'lam = 0.5': dict(dh=0.02, lam=0.5), 'lam = 2.0': dict(dh=0.02, lam=2.0),
              'k = 1e-11': dict(dh=0.02, log_k=-11.0), 'k = 1e-10': dict(dh=0.02, log_k=-10.0)}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for nom, p in variations.items():
    sim, q, Pe = simuler(**p)
    axes[0].plot(sim['Time'] / 86400, sim['Temp2'], label=f"{nom} (Pe={Pe:+.2f})")
    axes[1].plot(sim['Time'] / 86400, sim['Temp1'], label=nom)
for ax, titre in zip(axes, ['Capteur a 20 cm', 'Capteur a 10 cm']):
    ax.set_title(titre); ax.set_xlabel('Temps [jours]'); ax.grid(alpha=0.3)
axes[0].set_ylabel('Temperature [°C]'); axes[0].legend(fontsize=8)
plt.tight_layout(); plt.show()


## 7. Questions

1. Pour un regime infiltrant, pourquoi la temperature a 40 cm suit-elle davantage la surface que dans le cas conductif ?
2. Pour l'exfiltration, quel est le sens de propagation du signal ? Pourquoi la base (10 °C) pese-t-elle davantage ?
3. Changer `dz` : que devient le nombre de mailles, et la position des capteurs ? Pourquoi le resultat change-t-il peu si `dz` reste petit devant la longueur caracteristique du signal ?
4. Pour quelle valeur de $|q|$ le nombre de Peclet atteint-il 1 avec les parametres par defaut ?
